<a href="https://colab.research.google.com/github/DiegoTapia29/Tarea01_ManejadoresDeData/blob/main/Tarea_01ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from abc import ABC, abstractmethod
from typing import List

# =============================================================================
# 1. EXCEPCIONES PERSONALIZADAS (Manejo Robusto de Errores)
# =============================================================================
class AseguradoraError(Excepcion):
    """Excepción base para el dominio de la aseguradora."""
    pass

class EdadInvalidaError(AseguradoraError):
    """Arrojado cuando la edad ingresada o calculada está fuera del rango seteado de [18, 99]."""
    pass

class SumaAseguradaInvalidaError(AseguradoraError):
    """Arrojado cuando la suma asegurada no está en el rango [$500,000, $3,000,000]."""
    pass

class SexoInvalidoError(AseguradoraError):
    """Arrojado cuando el sexo no corresponde a 'M' o 'F'."""
    pass

class OpcionInvalidaError(AseguradoraError):
    """Arrojado cuando una opción binaria es distinta a 'Si' o 'No'."""
    pass

class TasaCambioInvalidaError(AseguradoraError):
    """Arrojado cuando la tasa de cambio es menor o igual a cero."""
    pass

# =============================================================================
# 2. VALIDACIÓN DE ENTRADAS (Principio de responsabilidad única)
# =============================================================================
class ValidadorEntrada:
    #Es la clase utilitaria con responsabilidad única de validar y parsear datos de consola.

    @staticmethod
    def validar_edad(val_str: str) -> int:
        try:
            edad = int(val_str)
        except ValueError:
            raise EdadInvalidaError("Debe ingresar un valor entero para la edad.")
        if not (18 <= edad <= 99):
            raise EdadInvalidaError("La edad debe ser un número entero entre 18 y 99 años.")
        return edad

    @staticmethod
    def validar_sexo(val_str: str) -> str:
        s = val_str.strip().upper()
        if s not in ['M', 'F']:
            raise SexoInvalidoError("El sexo debe ser 'M' (Masculino) o 'F' (Femenino).")
        return s

    @staticmethod
    def validar_si_no(val_str: str, campo: str) -> str:
        v = val_str.strip().capitalize()
        if v not in ["Si", "Sí", "No"]:
            raise OpcionInvalidaError(f"La respuesta dada '{campo}' debe ser 'Si' o 'No'.")
        return "Si" if v in ["Si", "Sí"] else "No"

    @staticmethod
    def validar_suma_asegurada(val_str: str) -> float:
        try:
            limpiar = val_str.replace(",", "").replace("$", "").strip()
            sa = float(limpiar)
        except ValueError:
            raise SumaAseguradaInvalidaError("Debe ingresar un monto válido.")
        if not (500000.0 <= sa <= 3000000.0):
            raise SumaAseguradaInvalidaError("La suma asegurada debe estar entre $500,000 y $3,000,000 MXN.")
        return sa


# =============================================================================
# 3. ESTRATEGIAS PARA EL FACTOR DE EDAD (Open/Closed y sustitución Liskov)
# =============================================================================
class Factor_edad(ABC):
    """Interfaz abstracta para calcular el factor de edad K."""

    @abstractmethod
    def obtener_factor(self, edad_ajustada: int) -> float:
        pass


class FactorFemenino(Factor_edad):
    #Cálculo de factor K para mujeres.

    def obtener_factor(self, edad_ajustada: int) -> float:
        if 18 <= edad_ajustada <= 25:
            return 1.5
        elif 25 < edad_ajustada <= 45:
            return 1.7
        elif 45 < edad_ajustada <= 65:
            return 2.0
        elif 65 < edad_ajustada <= 99:
            return 2.2
        raise EdadInvalidaError(f"Edad ajustada fuera de rango [18, 99]: {edad_ajustada}")


class FactorMasculino(Factor_edad):
    """Cálculo de factor K para hombres."""

    def obtener_factor(self, edad_ajustada: int) -> float:
        if 18 <= edad_ajustada <= 25:
            return 2.0
        elif 25 < edad_ajustada <= 45:
            return 2.3
        elif 45 < edad_ajustada <= 65:
            return 2.5
        elif 65 < edad_ajustada <= 99:
            return 3.0
        raise EdadInvalidaError(f"Edad fuera de rango [18, 99]: {edad_ajustada}")


# ============================================================================================
# 4. SERVICIO DE CONVERSIÓN DE MONEDA (Inversión de dependencias y Segregación de interfacesn)
# ============================================================================================
class ServicioMonedaInterface(ABC):
    #Interfaz abstracta para obtener la tasa de cambio de divisa

    @abstractmethod
    def obtener_tasa_usd(self) -> float:
        pass


class ServicioTasaCambioFija(ServicioMonedaInterface):
    #Simulación de servicio externo de divisas (tasa efectuada : 16.95 MXN/USD).

    def __init__(self, tasa: float = 16.95):
        if tasa <= 0:
            raise TasaCambioInvalidaError("La tasa de cambio debe ser estrictamente mayor a 0.")
        self._tasa = tasa

    def obtener_tasa_usd(self) -> float:
        return self._tasa


# =============================================================================
# MODELO DE DOMINIO Y CALCULADORA
# =============================================================================
class Asegurado:

    def __init__(self, id_asegurado: int, nombre: str, edad: int, sexo: str,
                 es_fumador: str, extra_prima: str, suma_asegurada: float):
        self.id_asegurado = id_asegurado
        self.nombre = nombre
        self.edad = edad
        self.sexo = sexo
        self.es_fumador = es_fumador
        self.extra_prima = extra_prima
        self.suma_asegurada = suma_asegurada

        self.edad_ajustada: int = 0
        self.factor_k: float = 0.0
        self.prima_mxn: float = 0.0
        self.prima_usd: float = 0.0

    def calcular_edad_ajustada(self) -> int:
        #Aplica las reglas de ajuste sobre la edad base delimitando a [18, 99].
        ajuste = 0
        if self.es_fumador == "No":
            ajuste -= 5
        if self.sexo == "F":
            ajuste -= 10
        if self.extra_prima == "Si":
            ajuste += 10

        edad_calculada = self.edad + ajuste
        self.edad_ajustada = max(18, min(99, edad_calculada))
        return self.edad_ajustada

    def generar_carnet(self) -> str:
        """Genera el reporte formateado en texto plano del carnet individual."""
        return (
            f"========================================\n"
            f"CARNET DE ASEGURADO #{self.id_asegurado}\n"
            f"========================================\n"
            f"Nombre:           {self.nombre}\n"
            f"Edad Base:        {self.edad} años\n"
            f"Sexo:             {self.sexo}\n"
            f"Fumador:          {self.es_fumador}\n"
            f"Extra-Prima:      {self.extra_prima}\n"
            f"Suma Asegurada:   ${self.suma_asegurada:,.2f} MXN\n"
            f"----------------------------------------\n"
            f"Edad Ajustada:    {self.edad_ajustada} años\n"
            f"Factor K:         {self.factor_k}\n"
            f"Prima Anual MXN:  ${self.prima_mxn:,.2f} MXN\n"
            f"Prima Anual USD:  ${self.prima_usd:,.2f} USD\n"
            f"========================================\n"
        )


class CalculadoraSeguro:
    """Calculadora responsable únicamente de realizar el cálculo de primas."""

    def __init__(self, servicio_moneda: ServicioMonedaInterface):
        self._servicio_moneda = servicio_moneda

    def _obtener_estrategia_factor(self, sexo: str) -> Factor_edad:
        if sexo == 'F':
            return FactorFemenino()
        elif sexo == 'M':
            return FactorMasculino()
        raise SexoInvalidoError(f"Sexo no validado(No binario es inválido): {sexo}")

    def procesar_asegurado(self, asegurado: Asegurado) -> None:
        """Calcula la edad ajustada, factor K, prima anual en MXN y USD."""
        edad_ajustada = asegurado.calcular_edad_ajustada()
        estrategia = self._obtener_estrategia_factor(asegurado.sexo)

        asegurado.factor_k = estrategia.obtener_factor(edad_ajustada)

        # Fórmula: P = (SA * K) / 1000
        asegurado.prima_mxn = (asegurado.suma_asegurada * asegurado.factor_k) / 1000.0

        tasa = self._servicio_moneda.obtener_tasa_usd()
        asegurado.prima_usd = asegurado.prima_mxn / tasa


# =============================================================================
# 5. PERSISTENCIA DE DATOS
# =============================================================================
class ExportadorCarnetInterface(ABC):
    """Interfaz abstracta para la persistencia de carnets."""

    @abstractmethod
    def exportar(self, asegurados: List[Asegurado], ruta_archivo: str) -> None:
        pass

class ExportadorCarnetTexto(ExportadorCarnetInterface):
    """Persiste los carnets de los asegurados en un archivo de texto."""

    def exportar(self, asegurados: List[Asegurado], ruta_archivo: str) -> None:
        try:
            with open(ruta_archivo, "w", encoding="utf-8") as f:
                for asegurado in asegurados:
                    f.write(asegurado.generar_carnet())
                    f.write("\n")
            print(f"\n[Éxito] Carnets exportados correctamente a '{ruta_archivo}'.")
        except (IOError, OSError, PermissionError) as e:
            print(f"\n[Error de Persistencia] No se pudo escribir en el archivo '{ruta_archivo}': {e}")

# =============================================================================
# REPORTES Y CAPTURA INTERACTIVA
# =============================================================================
class GeneradorReporteLote:
    """Procesa el lote completo de asegurados para emitir métricas estadísticas."""

    @staticmethod
    def mostrar_reporte(asegurados: List[Asegurado]) -> None:
        if not asegurados:
            print("No se procesó ningún asegurado.")
            return

        primas_mxn = [a.prima_mxn for a in asegurados]
        promedio = sum(primas_mxn) / len(primas_mxn)
        maxima = max(primas_mxn)
        minima = min(primas_mxn)

        # Filtrar únicamente los asegurados con extra-prima
        con_extra_prima = [a for a in asegurados if a.extra_prima == "Si"]

        print("\n" + "=" * 45)
        print("          REPORTE GENERAL DEL LOTE           ")
        print("=" * 45)
        print(f"Total de asegurados procesados: {len(asegurados)}")
        print(f"Prima Anual Promedio:           ${promedio:,.2f} MXN")
        print(f"Prima Anual Máxima:             ${maxima:,.2f} MXN")
        print(f"Prima Anual Mínima:             ${minima:,.2f} MXN")
        print("-" * 45)

        if con_extra_prima:
            top_ep = max(con_extra_prima, key=lambda x: x.prima_mxn)
            print("Asegurado con la Extra-Prima más alta:")
            print(f"  ID: {top_ep.id_asegurado} | Nombre: {top_ep.nombre}")
            print(f"  Prima: ${top_ep.prima_mxn:,.2f} MXN (${top_ep.prima_usd:,.2f} USD)")
        else:
            print("Ningún asegurado en el lote cuenta con Extra-Prima ('Si').")
        print("=" * 45 + "\n")


def capturar_asegurado_interactivo(id_asegurado: int) -> Asegurado:
    """Captura de datos por consola con ciclo de reintento por cada campo."""
    print(f"\n--- Captura de Datos para Asegurado #{id_asegurado} ---")
    nombre = input("Nombre del asegurado: ").strip()

    # Captura de Edad
    while True:
        try:
            val = input("Edad (18 - 99 años): ")
            edad = ValidadorEntrada.validar_edad(val)
            break
        except EdadInvalidaError as e:
            print(f"  [Error] {e}. Intente nuevamente.")

    # Captura de Sexo
    while True:
        try:
            val = input("Sexo ('M' para Masculino, 'F' para Femenino): ")
            sexo = ValidadorEntrada.validar_sexo(val)
            break
        except SexoInvalidoError as e:
            print(f"  [Error] {e}. Intente nuevamente.")

    # Captura de Fumador
    while True:
        try:
            val = input("¿Es fumador? ('Si' / 'No'): ")
            es_fumador = ValidadorEntrada.validar_si_no(val, "Fumador")
            break
        except OpcionInvalidaError as e:
            print(f"  [Error] {e}. Intente nuevamente.")

    # Captura de Extra-Prima
    while True:
        try:
            val = input("¿Tiene Extra-Prima? ('Si' / 'No'): ")
            extra_prima = ValidadorEntrada.validar_si_no(val, "Extra-Prima")
            break
        except OpcionInvalidaError as e:
            print(f"  [Error] {e}. Intente nuevamente.")

    # Captura de Suma Asegurada
    while True:
        try:
            val = input("Suma Asegurada anual ($500,000 - $3,000,000 MXN): ")
            suma_asegurada = ValidadorEntrada.validar_suma_asegurada(val)
            break
        except SumaAseguradaInvalidaError as e:
            print(f"  [Error] {e}. Intente nuevamente.")

    return Asegurado(id_asegurado, nombre, edad, sexo, es_fumador, extra_prima, suma_asegurada)

def main():
    print("==================================================")
    print("   SISTEMA DE CÁLCULO DE PRIMAS - ASEGURADORA   ")
    print("==================================================")

    # Inyección de dependencias
    servicio_moneda = ServicioTasaCambioFija(tasa=21.13)
    calculadora = CalculadoraSeguro(servicio_moneda)
    exportador = ExportadorCarnetTexto()

    # Obtener número de asegurados N
    while True:
        try:
            n_str = input("Ingrese el número de asegurados a procesar (N): ")
            n = int(n_str)
            if n <= 0:
                print("  [Error] El número de asegurados debe ser mayor a 0.")
                continue
            break
        except ValueError:
            print("  [Error] Debe ingresar un entero válido.")

    lote_asegurados: List[Asegurado] = []

    # Captura y procesamiento por lotes
    for i in range(1, n + 1):
        asegurado = capturar_asegurado_interactivo(i)
        calculadora.procesar_asegurado(asegurado)
        lote_asegurados.append(asegurado)

    # Generación de informe y persistencia
    GeneradorReporteLote.mostrar_reporte(lote_asegurados)
    exportador.exportar(lote_asegurados, "carnets_asegurados.txt")


if __name__ == "__main__":
    main()

   SISTEMA DE CÁLCULO DE PRIMAS - ASEGURADORA   
Ingrese el número de asegurados a procesar (N): 5

--- Captura de Datos para Asegurado #1 ---
Nombre del asegurado: Diego
Edad (18 - 99 años): 45
Sexo ('M' para Masculino, 'F' para Femenino): M
¿Es fumador? ('Si' / 'No'): no
¿Tiene Extra-Prima? ('Si' / 'No'): No
Suma Asegurada anual ($500,000 - $3,000,000 MXN): 600000

--- Captura de Datos para Asegurado #2 ---
Nombre del asegurado: A
Edad (18 - 99 años): 45
Sexo ('M' para Masculino, 'F' para Femenino): m
¿Es fumador? ('Si' / 'No'): si
¿Tiene Extra-Prima? ('Si' / 'No'): si
Suma Asegurada anual ($500,000 - $3,000,000 MXN): 800000

--- Captura de Datos para Asegurado #3 ---
Nombre del asegurado: b
Edad (18 - 99 años): 70
Sexo ('M' para Masculino, 'F' para Femenino): f
¿Es fumador? ('Si' / 'No'): si
¿Tiene Extra-Prima? ('Si' / 'No'): si
Suma Asegurada anual ($500,000 - $3,000,000 MXN): 900000

--- Captura de Datos para Asegurado #4 ---
Nombre del asegurado: f
Edad (18 - 99 años): 75
Sexo ('